# TDSP exp8 — does forced-move knowledge help RL? (phase 1: two-arm training)

Phase 0 (done locally, committed in `results/tdsp/exp8/`) already showed: on sparse
leaf-rich graphs (TREE, BA1) the forced set F is ~30% of vertices, and seeding it
preserves solution quality while cutting the agent's decisions by 66-88%.

**This notebook runs phase 1 — 2 trainings + 2 experiments:**

| # | cell | what it does | output on Drive |
|---|---|---|---|
| run 1 | §4 | **arm A** — trained on TREE n=20, plain env (no forcing), exp5 protocol | `armA_sparse_base/` |
| run 2 | §5 | **arm B** — same, but every episode seeded with F (`--force_leaf`) | `armB_sparse_forced/` |
| exp 3 | §7 | final eval: arms ±F on sparse (n≤200) + cross-family on dense/lattices | `eval_exp8/phase1_eval.*` |
| exp 4 | §8 | convergence A/B from the two `log.csv` curves (+ png plot for the thesis) | `eval_exp8/phase1_convergence.*` |

Both trainings use the exp5 protocol (n=20, 200k episodes, reward `size`, corrected
architecture) so the learning curves are comparable and *forcing is the only
difference* between the arms.

**If the runtime disconnects:** re-run §0-§3, then re-run the training cell of the
unfinished run — it **resumes** from the last checkpoint on Drive automatically.


## 0. Check the GPU is attached

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.'

## 1. Mount Google Drive (checkpoints + results survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_ROOT = '/content/drive/MyDrive/tdsp_runs/exp8'   # <-- everything for exp8 lives here
os.makedirs(RUN_ROOT, exist_ok=True)
print('exp8 outputs will live in', RUN_ROOT)

## 2. Get the code

Branch `tdsp-exp8-forcing` carries the exp8 code (sparse generators, forced-seed env,
`exp8_forcing.py`). Pick **github** (repo pushed) or **zip** (upload without git).

In [ ]:
USE = 'github'   # 'github' or 'zip'
REPO   = 'https://github.com/mahmoud-yousefi/files.git'
BRANCH = 'tdsp-exp8-forcing'

if USE == 'github':
    !git clone -b {BRANCH} {REPO} /content/files
else:
    from google.colab import files as _f
    up = _f.upload()
    import zipfile, os
    os.makedirs('/content/files', exist_ok=True)
    name = list(up)[0]
    zipfile.ZipFile(name).extractall('/content/files')

CODE = '/content/files/dsp_rl'
%cd $CODE
!ls

## 3. Install dependencies

In [ ]:
!pip -q install networkx scipy matplotlib
import torch, networkx, scipy, numpy
print('ok | torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3.5 Status check (safe to run any time -- reads Drive, starts nothing)

After a kernel reset / reconnect, run sections 0-3, then this cell: it tells you
exactly where each arm stands (last episode, validation |S|, wall-clock hours,
whether `model.pt`/`ckpt.pt` exist). Nothing here retrains.

In [ ]:
import csv, os
assert os.path.isdir(RUN_ROOT), 'run section 1 first (mount Drive + RUN_ROOT)'
for arm in ('armA_sparse_base', 'armB_sparse_forced'):
    d = f'{RUN_ROOT}/{arm}'
    lg = f'{d}/log.csv'
    if os.path.exists(lg):
        last = list(csv.DictReader(open(lg)))[-1]
        print(f"{arm}: last episode {last['episode']}, val|S| {last['val_mean_size']}, "
              f"elapsed {float(last['elapsed_s'])/3600:.2f} h, "
              f"model.pt={os.path.exists(f'{d}/model.pt')}, ckpt.pt={os.path.exists(f'{d}/ckpt.pt')}")
    else:
        print(f'{arm}: nothing on Drive yet')

## 4. Run 1 — arm A: training WITHOUT forcing (the control)

`--graph tree` switches the training family to uniform random labeled trees
(sparse, leaf-rich: mean ~7.5 leaves at n=20, so F is never empty).
Everything else is the exp5 protocol. Re-running this cell **resumes**.

In [ ]:
OUT_A = f'{RUN_ROOT}/armA_sparse_base'
EPISODES = 200000

!python train.py --env tdsp --reward size --graph tree \
    --episodes {EPISODES} --seed 0 --device cuda \
    --eval_every 5000 --val_graphs 50 --ckpt_every 10000 \
    --resume --out {OUT_A}

## 5. Run 2 — arm B: training WITH forced seeding (`--force_leaf`)

Identical to arm A except every episode starts with S = F (the leaf supports --
members of *every* total dominating set, so optimality is preserved by
construction, DECISIONS T8). In its log, `val_mean_size` includes the forced
vertices while `val_mean_return` counts only the agent's own decisions
(the gap between them is exactly mean |F|). Re-running this cell **resumes**.

In [ ]:
OUT_B = f'{RUN_ROOT}/armB_sparse_forced'
EPISODES = 200000

!python train.py --env tdsp --reward size --graph tree --force_leaf \
    --episodes {EPISODES} --seed 0 --device cuda \
    --eval_every 5000 --val_graphs 50 --ckpt_every 10000 \
    --resume --out {OUT_B}

## 6. (Optional) bring the exp5 dense model for a three-way comparison

`model.pt` files are not in git, so upload it once:
upload `dsp_rl/runs/tdsp_er20_s0_size_v2/model.pt` from your laptop into
Drive folder `MyDrive/tdsp_runs/exp8/base/` (the matching `config.json` comes
from the repo automatically). If you skip this, the eval simply compares the
two arms against greedy/GRASP/ILP.

In [ ]:
import os, shutil
BASE = f'{RUN_ROOT}/base_size_v2'
os.makedirs(BASE, exist_ok=True)
src = f'{RUN_ROOT}/base/model.pt'
if os.path.exists(src):
    shutil.copy('runs/tdsp_er20_s0_size_v2/config.json', f'{BASE}/config.json')
    shutil.copy(src, f'{BASE}/model.pt')
    print('base model staged at', BASE)
else:
    print('no base model uploaded -- eval will run with arms A/B only')

MODEL_ARGS = f'--model armA={OUT_A}/model.pt --model armB={OUT_B}/model.pt'
if os.path.exists(f'{BASE}/model.pt'):
    MODEL_ARGS += f' --model size_v2={BASE}/model.pt'
print(MODEL_ARGS)

## 7. Experiment 3 — final eval (the table for the thesis)

Each arm is evaluated **plain** and **with F seeded** (`+F` rows) on the sparse
families up to n=200, and plain on the dense/lattice families (cross-family
check that sparse training did not cost dense performance).
Writes `phase1_eval.{md,csv}` on Drive.

In [ ]:
# self-contained: safe to re-run after a kernel reset (never re-runs training)
import os
OUT_A = f'{RUN_ROOT}/armA_sparse_base'
OUT_B = f'{RUN_ROOT}/armB_sparse_forced'
EVAL_OUT = f'{RUN_ROOT}/eval_exp8'
BASE = f'{RUN_ROOT}/base_size_v2'
assert os.path.exists(f'{OUT_A}/model.pt'), 'arm A has no model.pt yet (section 4)'
assert os.path.exists(f'{OUT_B}/model.pt'), 'arm B has no model.pt yet (section 5)'

MODEL_ARGS = f'--model armA={OUT_A}/model.pt --model armB={OUT_B}/model.pt'
if os.path.exists(f'{BASE}/model.pt'):
    MODEL_ARGS += f' --model size_v2={BASE}/model.pt'
print(MODEL_ARGS)

!python experiments/tdsp/exp8_forcing.py eval {MODEL_ARGS} --reps 20 --out {EVAL_OUT}

## 8. Experiment 4 — convergence A/B (learning-curve comparison)

Compares the two `log.csv` curves: final `val_mean_size` and the first episode
that got within 0.05 of it (a proxy for learning speed). Writes
`phase1_convergence.csv` + a `.png` plot for the thesis.

In [ ]:
# self-contained: safe to re-run after a kernel reset
OUT_A = f'{RUN_ROOT}/armA_sparse_base'
OUT_B = f'{RUN_ROOT}/armB_sparse_forced'
EVAL_OUT = f'{RUN_ROOT}/eval_exp8'

!python experiments/tdsp/exp8_forcing.py convergence \
    --log armA={OUT_A}/log.csv --log armB={OUT_B}/log.csv --out {EVAL_OUT}
from IPython.display import Image, display
display(Image(f'{EVAL_OUT}/phase1_convergence.png'))

## 9. Zip the exp8 results and download

Copy-back map for the repo (do this on the laptop afterwards):

| on Drive (`MyDrive/tdsp_runs/exp8/...`) | into the repo |
|---|---|
| `eval_exp8/phase1_eval.{md,csv}` | `dsp_rl/results/tdsp/exp8/` |
| `eval_exp8/phase1_convergence.{csv,png}` + `phase1_convergence_summary.md` | `dsp_rl/results/tdsp/exp8/` |
| `armA_sparse_base/` and `armB_sparse_forced/` (`config.json`, `log.csv`) | `dsp_rl/runs/` (optional) |

In [ ]:
import os, zipfile
os.chdir('/content')
with zipfile.ZipFile('exp8_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for arm in ('armA_sparse_base', 'armB_sparse_forced'):
        for fn in ('config.json', 'log.csv'):
            p = f'{RUN_ROOT}/{arm}/{fn}'
            if os.path.exists(p):
                z.write(p, f'{arm}/{fn}')
    ev = f'{RUN_ROOT}/eval_exp8'
    if os.path.isdir(ev):
        for root, _, fns in os.walk(ev):
            for fn in fns:
                p = os.path.join(root, fn)
                z.write(p, os.path.relpath(p, RUN_ROOT))
!ls -la /content/exp8_results.zip
from google.colab import files as _f
_f.download('/content/exp8_results.zip')